In [ ]:
import os, time, copy, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
torch.manual_seed(42); np.random.seed(42)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('工作目录:', os.getcwd(), '| device:', DEV, '| torch', torch.__version__)

In [ ]:
# ============================================================
# 数据: NetCDF 9站 → 80/15/5 分割 → 各子集独立预处理 (与 xg_boost(3) 完全一致, 杜绝泄漏)
# LSTM 输入: (N, 168, F) 多变量历史窗 → 输出 (N, 48) 未来 PM2.5 (直接多步, 无递归)
# 每时间步特征 F=29 (LSTM 自身建模时序, 用瞬时特征, 不含滞后/滚动):
#   目标站 PM(1) + 其他8站 PM(8) + 气象(DEWP/HUMI/PRES/TEMP/Iws/precip=6)
#   + 额外气象(blh/msdwswrf/O3=3) + 时间周期(hour/month/dow sin·cos=6) + 风向one-hot(5) = 29
# 未来气象协变量: 保留完整 48h 逐小时气象序列 (9维×48步), 不作 mean/std 压缩
#   保留时间分辨率以捕捉边界层高度/风速/降水等逐小时动态
# 目标 = 线性 PM2.5 (expm1), 训练统计量标准化; 评估在线性 ug/m³ 空间
# 参考: https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html
# ============================================================
import netCDF4 as nc

N_ST = 9
WEATHER = ['DEWP','HUMI','PRES','TEMP','Iws','precipitation']
EXTRA   = ['blh','msdwswrf','O3']
TIME_COLS = ['hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos']
CBWD_COLS = ['cbwd_cv','cbwd_SE','cbwd_NW','cbwd_SW','cbwd_NE']
TS_COLS = ['pm_ave'] + [f'pm{i}' for i in range(1, N_ST)] + WEATHER + EXTRA + TIME_COLS + CBWD_COLS
WCOLS   = WEATHER + EXTRA                              # 9 维气象 (未来协变量, 保留完整48步序列)
WIDX    = [TS_COLS.index(c) for c in WCOLS]           # 气象列在 TS_COLS 中的索引
F_IN, L, H = len(TS_COLS), 168, 48                     # 输入特征数 / 回看168h / 预测48h
FUT_F, FUT_T = len(WCOLS), H                           # 未来气象: 9维 × 48步

def load_all(nc_path):
    """读取 9站 PM2.5 + 站0 全部气象/额外变量 (变量映射: K→°C, Pa→hPa, m→mm)"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    pm = np.asarray(f.variables['PM2.5'][:])
    temp = f.variables['t2m'][:,0]-273.15; dewp = f.variables['d2m'][:,0]-273.15
    pres = f.variables['sp'][:,0]/100.0;  tp = f.variables['tp'][:,0]*1000.0
    u, v = f.variables['u100'][:,0], f.variables['v100'][:,0]
    iws = np.sqrt(u**2+v**2); wdir = np.degrees(np.arctan2(-u,-v))%360
    cbwd = np.where(iws<0.5,'cv',np.where(wdir<90,'NE',np.where(wdir<180,'SE',np.where(wdir<270,'SW','NW')))).astype('<U2')
    a,b = 17.625,243.04
    humi = np.clip(100*np.exp(a*dewp/(dewp+b+1e-10))/np.exp(a*temp/(temp+b+1e-10)),0,100)
    blh = f.variables['blh'][:,0]; swr = f.variables['msdwswrf'][:,0]; o3 = f.variables['O3'][:,0]
    f.close()
    d = {'pm_ave': pm[:,0]}
    for i in range(1,N_ST): d[f'pm{i}'] = pm[:,i]
    d.update({'DEWP':dewp,'HUMI':humi,'PRES':pres,'TEMP':temp,'cbwd':cbwd,'Iws':iws,
              'precipitation':tp,'blh':blh,'msdwswrf':swr,'O3':o3})
    df = pd.DataFrame(d, index=dt); df.index.name='datetime'
    return df.resample('1h').first()

def preprocess(df, forward_only=False):
    """预处理. forward_only=True(测试集)仅前向填充, 不用任何未来值."""
    d = df.copy()
    def fill(s): return s.ffill() if forward_only else s.interpolate(method='time',limit=168).ffill().bfill()
    d['pm_ave'] = np.log1p(fill(d['pm_ave']))
    for i in range(1,N_ST): d[f'pm{i}'] = np.log1p(fill(d[f'pm{i}']))
    cols = WEATHER + EXTRA
    d[cols] = d[cols].ffill() if forward_only else d[cols].interpolate(method='time').ffill().bfill()
    h = d.index.hour.values.astype(float)
    d['hour_sin']=np.sin(2*np.pi*h/24); d['hour_cos']=np.cos(2*np.pi*h/24)
    m = d.index.month.values.astype(float)
    d['month_sin']=np.sin(2*np.pi*m/12); d['month_cos']=np.cos(2*np.pi*m/12)
    dw = d.index.dayofweek.values.astype(float)
    d['dow_sin']=np.sin(2*np.pi*dw/7); d['dow_cos']=np.cos(2*np.pi*dw/7)
    d['cbwd'] = d['cbwd'].ffill().bfill()
    d['cbwd'] = pd.Categorical(d['cbwd'], categories=['cv','SE','NW','SW','NE'])
    d = pd.concat([d.drop('cbwd',axis=1), pd.get_dummies(d['cbwd'],prefix='cbwd').astype(float)], axis=1)
    return d

# 加载 + 分割 + 预处理 (先分割再处理, 测试集前向填充)
df_raw = load_all('data3/dataset_yrd.nc')
n = len(df_raw); n1 = int(n*0.80); n2 = int(n*0.95)
df_tr = preprocess(df_raw.iloc[:n1])
df_va = preprocess(df_raw.iloc[n1:n2])
df_te = preprocess(df_raw.iloc[n2:], forward_only=True)

# 标准化: 仅用训练统计量 (无泄漏)
Xm = df_tr[TS_COLS].values.astype(np.float32)
FEAT_MEAN = Xm.mean(axis=0); FEAT_STD = Xm.std(axis=0) + 1e-8
def std_feats(df): return ((df[TS_COLS].values.astype(np.float32) - FEAT_MEAN) / FEAT_STD)
Ftr, Fva, Fte = std_feats(df_tr), std_feats(df_va), std_feats(df_te)
# 目标 = 线性 PM2.5, 用训练统计量标准化
PMtr = np.expm1(df_tr['pm_ave'].values).astype(np.float32)
Y_MEAN, Y_STD = float(PMtr.mean()), float(PMtr.std()) + 1e-8
def std_y(df): return ((np.expm1(df['pm_ave'].values).astype(np.float32) - Y_MEAN) / Y_STD)
Ytr, Yva, Yte = std_y(df_tr), std_y(df_va), std_y(df_te)
CLIP_HI = float(np.expm1(df_tr['pm_ave'].max()))

# 滑窗索引: 起点T∈[L, n-H], 训练子采样降重叠 (stride), 验证稀疏; 各窗内历史[T-L:T]与未来[T:T+H]同属一子集, 无跨集泄漏
TRAIN_STRIDE, VAL_STRIDE = 3, 6
def win_idx(nlen, stride):
    return np.arange(L, nlen - H + 1, stride, dtype=np.int64)
idx_tr = win_idx(len(df_tr), TRAIN_STRIDE)
idx_va = win_idx(len(df_va), VAL_STRIDE)
# 训练样本权重: 仅放大上四分位(峰值)样本, 其余=1 不失真
lvl_tr = np.array([Ytr[T:T+H].mean() for T in idx_tr])
q75, q99 = float(np.quantile(lvl_tr, 0.75)), float(np.quantile(lvl_tr, 0.99))
WT_TR = 1.0 + 1.0 * np.clip((lvl_tr - q75) / (q99 - q75 + 1e-8), 0.0, 1.0)

class WinDS(Dataset):
    """历史窗 (168,29) + 未来气象逐小时 (48,9) + 未来PM目标 (48,)"""
    def __init__(self, feat, y, idx, w=None):
        self.feat, self.y, self.idx, self.w = feat, y, idx, w
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        T = int(self.idx[i])
        x = torch.as_tensor(self.feat[T-L:T])               # (L, F_IN) 历史, 用至 T-1
        fut = torch.as_tensor(self.feat[T:T+H, WIDX])         # (H, 9) 未来气象逐小时(标准化, 预报可得)
        y = torch.as_tensor(self.y[T:T+H])                   # (H,) 未来 PM (标准化)
        if self.w is not None: return x, fut, y, torch.as_tensor(self.w[i], dtype=torch.float32)
        return x, fut, y

BS = 256
tr_loader = DataLoader(WinDS(Ftr, Ytr, idx_tr, WT_TR), batch_size=BS, shuffle=True)
va_loader = DataLoader(WinDS(Fva, Yva, idx_va), batch_size=512, shuffle=False)
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)} | 特征/步 {F_IN} | 未来气象(逐时) {FUT_F}×{FUT_T}')
print(f'滑窗样本: 训练 {len(idx_tr)} (stride={TRAIN_STRIDE}) | 验证 {len(idx_va)} (stride={VAL_STRIDE})')
print(f'目标(线性) mean={Y_MEAN:.1f} std={Y_STD:.1f} | 预测上限 {CLIP_HI:.0f} | 权重[{WT_TR.min():.2f},{WT_TR.max():.2f}]')

In [3]:
# ============================================================
# 模型: biLSTM编码168h历史 → 注意力池化; 第二层LSTM编码未来48h气象 → 融合 → MLP直出48步
# 损失: 加权MSE + 一阶差分损失(导数损失) + 动态峰值加权, 强制曲线动起来
# 参考: https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html
# ============================================================
HIDDEN, N_LAYERS, DROP = 128, 2, 0.2

class LSTMForecaster(nn.Module):
    def __init__(self, n_in, hidden, n_layers, fut_f, fut_t, n_out, drop=0.2):
        super().__init__()
        # 编码器: 双向LSTM编码历史
        self.enc = nn.LSTM(n_in, hidden, n_layers, batch_first=True, bidirectional=True, dropout=drop)
        self.attn = nn.Linear(hidden*2, 1)              # 历史加性注意力打分
        # 未来气象编码器: 第二层LSTM, 编码48步气象序列 (保留逐小时时间分辨率)
        self.fut_enc = nn.LSTM(fut_f, hidden, n_layers, batch_first=True, bidirectional=True, dropout=drop)
        self.fut_attn = nn.Linear(hidden*2, 1)          # 未来气象注意力打分
        self.drop = nn.Dropout(drop)
        # 融合头: [历史context(2H) + 未来气象context(2H)] → MLP
        self.head = nn.Sequential(
            nn.Linear(hidden*4, 384), nn.GELU(), nn.Dropout(drop),
            nn.Linear(384, 192), nn.GELU(), nn.Dropout(drop), nn.Linear(192, n_out))
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def _attn_pool(self, lstm_out, score_layer):
        """加性注意力池化: lstm_out (B,T,2H) → context (B,2H)"""
        w = torch.softmax(score_layer(lstm_out).squeeze(-1), dim=1)   # (B,T)
        return (lstm_out * w.unsqueeze(-1)).sum(1)                    # (B,2H)
    def forward(self, x, fut):                          # x:(B,L,F) fut:(B,FUT_T,FUT_F)
        enc_out, _ = self.enc(x)                        # (B,L,2H)
        hist_ctx = self._attn_pool(enc_out, self.attn)  # (B,2H)
        fut_out, _ = self.fut_enc(fut)                  # (B,FUT_T,2H)
        fut_ctx = self._attn_pool(fut_out, self.fut_attn)  # (B,2H)
        ctx = torch.cat([hist_ctx, fut_ctx], dim=1)     # (B,4H)
        return self.head(self.drop(ctx))                # (B, n_out=48)

model = LSTMForecaster(F_IN, HIDDEN, N_LAYERS, FUT_F, FUT_T, H, DROP).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')

# ============================================================
# 损失: 加权MSE + 一阶差分损失 + 动态峰值加权
# L = w·MSE + λ_diff·|Δpred - Δy| + 0.5·(1+peak_mask)·MSE
#   一阶差分损失: 惩罚预测与真实变化率的失配, 抑制曲线平缓
#   动态峰值加权: 真实高值时段(>q75)额外放大权重
# ============================================================
LAMBDA_DIFF = 0.5          # 一阶差分损失权重
LAMBDA_PEAK = 1.0          # 动态峰值加权强度
Y_Q75 = float(np.quantile(Ytr, 0.75))             # 训练目标75分位(标准化空间), 用于峰值时段判定

def dyn_loss(pred, y, w):
    """pred,y: (B,48)标准化; w: (B,)样本权重. 返回标量损失."""
    # 1) 加权 MSE
    mse = (w * ((pred - y)**2).mean(dim=1)).mean()
    # 2) 一阶差分损失 (导数损失): |Δpred - Δy|, 强制变化率匹配
    dp = pred[:, 1:] - pred[:, :-1]                      # (B,47) 预测差分
    dy = y[:, 1:] - y[:, :-1]                            # (B,47) 真实差分
    diff_loss = (w.unsqueeze(1) * (dp - dy).abs()).mean()
    # 3) 动态峰值加权: 真实高值时段(y>q75)额外放大权重
    peak_mask = (y > Y_Q75).float()                     # (B,48) 1=峰值时段
    peak_w = 1.0 + LAMBDA_PEAK * peak_mask               # (B,48) 峰值处权重2
    peak_loss = (peak_w * (pred - y)**2).mean()
    return mse + LAMBDA_DIFF * diff_loss + 0.5 * peak_loss

@torch.no_grad()
def eval_val():
    model.eval(); P, A = [], []
    for xb, fb, yb in va_loader:
        p = model(xb.to(DEV), fb.to(DEV)).cpu().numpy()
        P.append(p); A.append(yb.numpy())
    P = np.concatenate(P) * Y_STD + Y_MEAN; A = np.concatenate(A) * Y_STD + Y_MEAN
    return float(np.sqrt(((P - A)**2).mean()))

EPOCHS, PATIENCE = 30, 5
best_val, best_state, bad = 1e9, None, 0
t0 = time.time()
for ep in range(1, EPOCHS+1):
    model.train(); tl = []
    for xb, fb, yb, wb in tr_loader:
        xb, fb, yb, wb = xb.to(DEV), fb.to(DEV), yb.to(DEV), wb.to(DEV)
        opt.zero_grad(); loss = dyn_loss(model(xb, fb), yb, wb); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪防爆炸
        opt.step(); tl.append(loss.item())
    vr = eval_val(); sched.step(vr)
    if vr < best_val - 1e-3:
        best_val, best_state, bad = vr, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
    print(f'epoch {ep:2d} | train_loss {np.mean(tl):.4f} | val_RMSE(linear) {vr:.2f} | best {best_val:.2f} | bad {bad}/{PATIENCE} | lr {opt.param_groups[0]["lr"]:.1e}', flush=True)
    if bad >= PATIENCE: print('  early stop'); break
print(f'\n训练完成 {time.time()-t0:.0f}s | 最佳 val RMSE(linear) = {best_val:.2f}')
model.load_state_dict(best_state); model.eval()

# per-horizon 偏差校正 (仅验证集估计, 无泄漏)
@torch.no_grad()
def val_pred():
    P, A = [], []
    for xb, fb, yb in va_loader:
        P.append(model(xb.to(DEV), fb.to(DEV)).cpu().numpy()); A.append(yb.numpy())
    return np.concatenate(P)*Y_STD+Y_MEAN, np.concatenate(A)*Y_STD+Y_MEAN
P_va, A_va = val_pred()
BIAS_H = (P_va - A_va).mean(axis=0)                  # (48,) 每个 horizon 的系统偏差
ALPHA = 0.5
print(f'验证 per-horizon bias: h1={BIAS_H[0]:+.2f} h24={BIAS_H[23]:+.2f} h48={BIAS_H[47]:+.2f} (校正系数 α={ALPHA})')

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 评估: 36 组, 每组 168h 历史 + 未来48h气象 → 一次预测 48h (直接多步, 无递归)
# 偏差校正减 α·bias_h; 截断[0, CLIP_HI]; 画图 12×3
# 指标: 分组平均 RMSE(48h|24h) + 池化(MAE/FAC2/R/峰值) + 持久性基线
# ============================================================
N_GROUPS, N_PRED = 36, 48
STRIDE = (len(df_te) - L - N_PRED) // (N_GROUPS - 1)
PEAK = 75.0

@torch.no_grad()
def predict_groups():
    P, A = [], []
    for g in range(N_GROUPS):
        T = g * STRIDE + L                            # 预测起点, 历史[T-L:T]用至T-1
        x = torch.as_tensor(Fte[T-L:T]).float().unsqueeze(0).to(DEV)      # (1,L,F)
        fut = torch.as_tensor(Fte[T:T+H, WIDX]).float().unsqueeze(0).to(DEV)  # (1,H,9)
        pred = model(x, fut).cpu().numpy().ravel() * Y_STD + Y_MEAN
        pred -= ALPHA * BIAS_H                        # per-horizon 偏差校正
        P.append(np.clip(pred, 0, CLIP_HI)); A.append(np.expm1(df_te['pm_ave'].values[T:T+N_PRED]))
    return P, A

def persist_groups():
    P, A = [], []
    pm = np.expm1(df_te['pm_ave'].values)
    for g in range(N_GROUPS):
        T = g * STRIDE + L; P.append(np.full(N_PRED, pm[T-1])); A.append(pm[T:T+N_PRED])
    return P, A

def group_rmse(P, A, k=None):
    return float(np.mean([np.sqrt(((p[:k]-a[:k])**2).mean()) for p, a in zip(P, A)]))

def pooled(P, A, k, tag):
    p = np.concatenate([x[:k] for x in P]); a = np.concatenate([x[:k] for x in A])
    rmse = np.sqrt(((p-a)**2).mean()); mae = np.abs(p-a).mean()
    fac2 = np.mean((p/(a+1e-6) <= 2) & (a/(p+1e-6) <= 2)); r = np.corrcoef(p, a)[0,1]
    pk = a > PEAK
    prmse = np.sqrt(((p[pk]-a[pk])**2).mean()) if pk.sum() else float('nan')
    pmae = np.abs(p[pk]-a[pk]).mean() if pk.sum() else float('nan')
    print(f'  [{tag}] RMSE={rmse:.2f} MAE={mae:.2f} FAC2={fac2:.3f} R={r:.3f} | 峰值(>{PEAK:.0f})n={int(pk.sum())} RMSE={prmse:.2f} MAE={pmae:.2f}')

P, A = predict_groups()
rmses = [np.sqrt(((p-a)**2).mean()) for p, a in zip(P, A)]
fig, axes = plt.subplots(12, 3, figsize=(18, 42)); axes = axes.flatten()
for g in range(N_GROUPS):
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, A[g], lw=0.8, label='actual'); ax.plot(h, P[g], lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[g]:.1f}', fontsize=8)
    ax.legend(fontsize=6); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'LSTM (biLSTM-enc + fut-weather-LSTM, diff-loss, peak-weighted) — {N_GROUPS} Groups', fontsize=12, y=1.005)
plt.tight_layout(); plt.show()

PP, AA = persist_groups()
print('=== 分组平均 RMSE (既定基准) ===')
print(f'  LSTM(直接多步) : 48h={group_rmse(P,A):.2f} | 24h={group_rmse(P,A,24):.2f}')
print(f'  持久性         : 48h={group_rmse(PP,AA):.2f} | 24h={group_rmse(PP,AA,24):.2f}')
print('\n=== 池化综合指标 (LSTM) ===')
pooled(P, A, 48, '48h'); pooled(P, A, 24, '24h')
print('=== 池化综合指标 (持久性) ===')
pooled(PP, AA, 48, '48h'); pooled(PP, AA, 24, '24h')
print('\n=== 各组 RMSE(48h|24h) ===')
for g in range(N_GROUPS):
    r48 = np.sqrt(((P[g]-A[g])**2).mean()); r24 = np.sqrt(((P[g][:24]-A[g][:24])**2).mean())
    print(f'  G{g+1:2d}: {r48:6.2f} | {r24:6.2f}')